# 003_team_mass: Владение по движению взвешенной массы команды
Запустите **Run All**. Прогресс показывает tqdm; результаты и промежуточный progress.json — в local_review/experiments/003_team_mass.
Нейросети не используются. Все исходные точки сохранены, координаты не печатаются. Геометрическая эвристика — гипотеза о владении, а не правило футбола.

Перебор задан заранее в grid.json. Выбираем максимум spellF1 на первых 4 минутах, при равенстве macroF1, затем меньший номер варианта. Последняя минута оценивается только для выбранной конфигурации. В обоих случаях история алгоритма идёт с начала тайма: контекст до проверки разрешён, будущие кадры не используются. Первые 10 секунд тайма используются для определения направления сторон; это явно оговорённая калибровка, а не полностью потоковый старт.

На публичных 5 минутах всего 17 владений, а вариантов много: выигрыш на обучении ожидаемо оптимистичен. Проверочная минута ранее использовалась в проекте, поэтому она тоже не независимый тест. Не используйте её для нового массового подбора.

D определяется неизменённой эвристикой эксперимента 001: положение/движение мяча, медленное движение игроков, автоматические интервалы стандартов. Поэтому «только масса» означает источник выбора H/A; D всё ещё использует мяч и события. При отсутствии уверенного начального сигнала выдаём X; после первого решения сохраняем прошлую команду до подтверждения обратного сигнала.


In [ ]:
from pathlib import Path
import sys, json, time, warnings, hashlib, itertools, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0,str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from starter.load_smart_tagging import load as load_smart_tagging
from evaluation import possession_metrics as pm
EXPERIMENT = "003_team_mass"
OUT = ROOT / "local_review/experiments" / EXPERIMENT
OUT.mkdir(parents=True,exist_ok=True)
ignore = ROOT / "local_review/.gitignore"
if not ignore.exists(): ignore.write_text("*\n")
CFG = json.loads((ROOT / "sol/001_box5_vote100/config.json").read_text())
GRID = json.loads((ROOT / "sol" / EXPERIMENT / "grid.json").read_text())
N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
PITCH = np.array(GAMES[GAME]["pitch"],dtype=np.float64)
BLOCKS = half_video_blocks(GAME)
PATHS = asset_paths()

def save_json(path, value):
    temp = path.with_suffix(path.suffix+".tmp")
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False))
    temp.replace(path)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

started=time.perf_counter()
print(EXPERIMENT, "| Results:", OUT)


## Исходные наблюдения и общий детектор остановок
Код D повторяет 001; параметры не подбираются в этих экспериментах.

In [ ]:
# Load only ATD positions; raw speed units are inconsistent in the source documentation.
# Читаем только координаты ATD: единицы готовой скорости неоднозначны.
track_teams = team_of_each_track(PATHS["atd_meta"])
players = np.full((N, len(track_teams), 2), np.nan, dtype=np.float64)
ball = np.full((N, 2), np.nan, dtype=np.float64)
read_count = 0
with open(PATHS["atd_raw"]) as handle:
    for i, line in enumerate(tqdm(handle,total=N,desc="Tracking / Трекинг")):
        frame_id, player_text, ball_text = line.strip().split(":")
        assert int(frame_id) == i + 1, "ATD/video alignment mismatch"
        player_values = np.fromstring(player_text.replace(";", ","), sep=",").reshape(-1, 3)
        assert len(player_values) == len(track_teams)
        players[i] = player_values[:, :2] * PITCH
        ball[i] = np.fromstring(ball_text, sep=",")[:2] * PITCH
        read_count += 1
assert read_count == N
ball_seen = np.isfinite(ball).all(axis=1)
players_seen = np.isfinite(players).all(axis=2)

# The window is an axis-aligned SQUARE, not a radius-2.5 m circle.
# Окно — КВАДРАТ вдоль осей поля, не круг радиусом 2.5 м.
def box_votes(player_xy, ball_xy, team_ids, width_m):
    inside = (np.isfinite(player_xy).all(axis=2)
              & np.isfinite(ball_xy).all(axis=1)[:, None]
              & (np.abs(player_xy - ball_xy[:, None, :]) <= width_m / 2).all(axis=2))
    home_count = (inside & (team_ids == 0)).sum(axis=1)
    away_count = (inside & (team_ids == 1)).sum(axis=1)
    # +1 means H, -1 means A, 0 means abstention for this frame.
    # +1 — H, -1 — A, 0 — нет голоса в этом кадре.
    votes = np.sign(home_count - away_count).astype(np.int8)
    return home_count, away_count, votes

home_count, away_count, frame_votes = box_votes(players, ball, track_teams, CFG["box_width_m"])
# A narrow synthetic check verifies corners, ties, and missing-ball handling.
# Проверяем на искусственных точках углы квадрата, ничью и отсутствие мяча.
test_players = np.array([[[2.5, 2.5], [2.6, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]])
test_ball = np.array([[0., 0.], [0., 0.], [np.nan, np.nan]])
assert box_votes(test_players, test_ball, np.array([0, 1]), 5)[2].tolist() == [1, 0, 0]

# A set-piece interval provides context only; its team tag is not used as a possession label.
# Интервал стандарта — только контекст; его Team не копируем в метку владения.
set_piece_context = np.zeros(N, dtype=bool)
for event in load_smart_tagging():
    if (event["code"] or "").strip().upper() == "SET PIECES":
        start = max(0, event["start_frame"])
        stop = min(N, event["end_frame"])
        set_piece_context[start:stop] = True
print("ATD and automatic set-piece context loaded; no ground truth accessed.")


In [ ]:
lag = CFG["motion_lag_frames"]
dt = lag / FPS
player_motion = np.full(N, np.nan)
ball_motion = np.full(N, np.nan)

# Calculate motion separately in each half, so halftime never creates a false displacement.
# Движение считаем отдельно внутри таймов, не соединяя их через перерыв.
for _, first, last in BLOCKS:
    stop = last + 1
    speed = np.linalg.norm(players[first + lag:stop] - players[first:stop - lag], axis=2) / dt
    speed[(speed > CFG["max_player_motion_mps"]) | ~np.isfinite(speed)] = np.nan
    enough = np.isfinite(speed).sum(axis=1) >= CFG["minimum_motion_tracks"]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="All-NaN slice encountered", category=RuntimeWarning)
        medians = np.nanmedian(speed, axis=1)
    medians[~enough] = np.nan
    player_motion[first + lag:stop] = medians
    seen = ball_seen[first:stop]
    cumulative = np.r_[0, np.cumsum(seen)]
    continuous = (cumulative[lag + 1:] - cumulative[:-(lag + 1)]) == lag + 1
    ball_speed = np.linalg.norm(ball[first + lag:stop] - ball[first:stop - lag], axis=1) / dt
    ball_speed[~continuous] = np.nan
    ball_motion[first + lag:stop] = ball_speed

outside = ball_seen & ((ball < -CFG["outside_margin_m"]) | (ball > PITCH + CFG["outside_margin_m"])).any(axis=1)
inside_field = ball_seen & (ball >= 0).all(axis=1) & (ball <= PITCH).all(axis=1)

def detect_dead_ball():
    dead = np.zeros(N, dtype=bool)
    # Diagnostic counts record triggers, not ground-truth stopping reasons.
    # Счётчики показывают срабатывания эвристик, а не истинные причины остановок.
    reasons = {"outside": 0, "stationary": 0, "missing_and_slow": 0}
    for _, first, last in BLOCKS:
        state = False
        outside_run = missing_run = stationary_run = slow_run = restart_ball_run = restart_players_run = 0
        for i in range(first, last + 1):
            outside_run = outside_run + 1 if outside[i] else 0
            missing_run = missing_run + 1 if not ball_seen[i] else 0
            stationary_run = stationary_run + 1 if ball_motion[i] <= CFG["stationary_ball_mps"] else 0
            slow_run = slow_run + 1 if player_motion[i] <= CFG["slow_players_mps"] else 0
            restart_ball_run = restart_ball_run + 1 if inside_field[i] and ball_motion[i] >= CFG["restart_ball_mps"] else 0
            restart_players_run = restart_players_run + 1 if player_motion[i] >= CFG["restart_players_mps"] else 0
            if not state:
                stationary_threshold = (CFG["set_piece_stationary_confirm_frames"] if set_piece_context[i]
                                        else CFG["stationary_ball_confirm_frames"])
                slow = slow_run >= CFG["slow_players_confirm_frames"]
                if outside_run >= CFG["outside_confirm_frames"]:
                    state = True
                    reasons["outside"] += 1
                elif slow and stationary_run >= stationary_threshold:
                    state = True
                    reasons["stationary"] += 1
                elif slow and missing_run >= CFG["missing_ball_confirm_frames"]:
                    state = True
                    reasons["missing_and_slow"] += 1
            elif not outside[i] and (restart_ball_run >= CFG["restart_ball_confirm_frames"]
                                    or restart_players_run >= CFG["restart_players_confirm_frames"]):
                state = False
                # Reset evidence so a previous stoppage cannot immediately trigger a new one.
                # Сбрасываем накопленные признаки предыдущей остановки.
                outside_run = missing_run = stationary_run = slow_run = 0
            dead[i] = state
    return dead, reasons

dead, dead_trigger_counts = detect_dead_ball()


## Движение массы
Для общих видимых слотов считаем середины их положений на концах окна. Вес равен `1 / (1 + (расстояние до медианы команды / scale)²)`, вес предполагаемого вратаря дополнительно умножается на 0.25.
Взвешенная сумма смещений / сумма весов / время — скорость центра одного и того же набора точек.
Слишком большие скорости (>10 м/с), координаты вне поля и пропуски исключаются только из расчёта. Нужны минимум 4 точки и половина наблюдаемых игроков.
Продольную скорость переводим в направление атаки каждой команды. Сигнал = скорость H в направлении её атаки − скорость A в направлении её атаки. Положительный устойчивый сигнал означает H, отрицательный — A. Поперечное движение влияет на веса и контроль скорости, но не на знак владения.
Сглаживание — среднее по предыдущим кадрам, переход требует порога и нескольких последовательных подтверждений. D и перерыв сбрасывают состояние. Правдоподобные подмены слотов могут оставаться; геометрия не гарантирует контролируемого владения.

In [ ]:
# Fixed geometry rules; no possession ground truth is consulted here.
# Фиксированные геометрические правила; разметка владения здесь не используется.
defending_left = {}
keeper_proxy = np.zeros(players_seen.shape,dtype=bool)
for half, first, last in BLOCKS:
    anchor={1:11185,2:113742}[half]
    centers=[]
    for team in (0,1):
        x=players[anchor:anchor+250,track_teams==team,0]
        centers.append(float(np.nanmedian(np.nanmedian(x,axis=1))))
    assert abs(centers[0]-centers[1]) > .05*PITCH[0]
    defending_left[half]=[centers[0]<centers[1],centers[1]<centers[0]]
    for team in (0,1):
        slots=np.flatnonzero(track_teams==team)
        x=players[first:last+1,:,0][:,slots].copy()
        valid=players_seen[first:last+1][:,slots]
        left=defending_left[half][team]
        x[~valid]=np.inf if left else -np.inf
        candidate=x.argmin(axis=1) if left else x.argmax(axis=1)
        rr=np.arange(len(x)); depth=x[rr,candidate] if left else PITCH[0]-x[rr,candidate]
        eligible=valid.any(axis=1)&(depth>=-1e-10)&(depth<=16.5+1e-10)
        keeper_proxy[first+rr[eligible],slots[candidate[eligible]]]=True

# Compare the SAME observed slots at both ends: raw centroid differences would
# confuse disappearing detections with team movement. Slots are not stable identities;
# reject impossible displacements, but plausible identity switches remain a limitation.
# Сравниваем одинаковые видимые слоты на концах окна: исчезновение игрока не движение.
# Слоты не гарантируют личность; отсекаем слишком большие смещения, но не все подмены.
def team_mass_velocity(lag, scale, limit):
    velocities=np.full((limit,2),np.nan)
    for half,first,last in BLOCKS:
        stop=min(last+1,limit)
        if first+lag>=stop: continue
        for team in (0,1):
            slots=np.flatnonzero(track_teams==team)
            # Process small batches to limit RAM / Небольшие блоки ограничивают память
            for a in range(first+lag,stop,2048):
                b=min(a+2048,stop)
                current=players[a:b][:,slots]
                previous=players[a-lag:b-lag][:,slots]
                speed=np.linalg.norm(current-previous,axis=-1)/(lag/FPS)
                valid=(np.isfinite(current).all(-1)&np.isfinite(previous).all(-1)&(speed<=10.)
                       &(current>=0).all(-1)&(current<=PITCH).all(-1)
                       &(previous>=0).all(-1)&(previous<=PITCH).all(-1))
                middle=(current+previous)/2
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore",RuntimeWarning)
                    center=np.nanmedian(np.where(valid[...,None],middle,np.nan),axis=1)
                distance=np.linalg.norm(middle-center[:,None],axis=-1)
                weights=np.where(valid,1/(1+(distance/scale)**2),0.)
                keeper=keeper_proxy[a:b][:,slots]|keeper_proxy[a-lag:b-lag][:,slots]
                weights*=np.where(keeper,.25,1.)
                total=weights.sum(axis=1)
                displacement=np.nan_to_num(current-previous,nan=0.)
                movement=(weights[...,None]*displacement).sum(axis=1)/np.maximum(total[:,None],1e-12)
                # At least four shared valid observations per team.
                # Нужно минимум четыре общих корректных наблюдения команды.
                observed_now=players_seen[a:b][:,slots].sum(axis=1)
                enough=(valid.sum(axis=1)>=4)&(valid.sum(axis=1)>=.5*observed_now)&(total>0)
                sign=1 if defending_left[half][team] else -1
                velocities[a:b,team]=np.where(enough,sign*movement[:,0]/(lag/FPS),np.nan)
    return velocities[:,0]-velocities[:,1]

def trailing_mean(values, window, dead_mask, blocks):
    # D and halftime reset accumulation. Missing samples do not become zero motion.
    # D и перерыв сбрасывают накопление. Пропуски не считаем нулевой скоростью.
    result=np.full(len(values),np.nan)
    for _,first,last in blocks:
        stop=min(last+1,len(values))
        if first>=stop: continue
        v=values[first:stop]; d=dead_mask[first:stop]
        ids=np.arange(len(v))
        reset=np.maximum.accumulate(np.where(d,ids+1,0))
        lo=np.maximum(np.maximum(0,ids-window+1),reset)
        good=np.isfinite(v)&~d
        sums=np.r_[0.,np.cumsum(np.where(good,v,0.))]
        counts=np.r_[0,np.cumsum(good)]
        total=counts[ids+1]-counts[lo]
        mean=(sums[ids+1]-sums[lo])/np.maximum(total,1)
        enough=(total>=np.maximum(1,np.ceil((ids+1-lo)*.5)))&~d
        result[first:stop]=np.where(enough,mean,np.nan)
    return result

def decode_signal(signal, dead_mask, blocks, threshold, confirm):
    # Hysteresis: keep the previous team until the opposite evidence persists.
    # Гистерезис: сохраняем команду, пока противоположный сигнал не подтвердится.
    result=np.full(len(signal),"X",dtype="<U1")
    for _,first,last in blocks:
        state="X"; candidate="X"; count=0
        for i in range(first,min(last+1,len(signal))):
            if dead_mask[i]:
                result[i]="D"; state="X"; candidate="X"; count=0
                continue
            wanted="H" if signal[i]>threshold else "A" if signal[i]<-threshold else "X"
            if wanted=="X" or wanted==state:
                candidate="X"; count=0
            else:
                count=count+1 if candidate==wanted else 1
                candidate=wanted
                if count>=confirm:
                    state=wanted; candidate="X"; count=0
            result[i]=state
    return result

# Synthetic checks protect reset rules, persistent evidence, and missing observations.
# Искусственные проверки сброса, подтверждения сигнала и пропусков.
assert decode_signal(np.array([1.,1.,-1.,-1.,np.nan]),np.zeros(5,bool),[(1,0,4)],.5,2).tolist()==["X","H","H","A","A"]
assert decode_signal(np.ones(3),np.array([False,True,False]),[(1,0,2)],.5,1).tolist()==["H","D","H"]
np.testing.assert_allclose(trailing_mean(np.array([1.,np.nan,3.]),2,np.zeros(3,bool),[(1,0,2)]),[1.,1.,3.])
assert np.isnan(trailing_mean(np.array([1.,1.,np.nan]),3,np.array([False,True,False]),[(1,0,2)])[2])
print("Geometry and temporal checks passed")

# Known translation must survive a disappearing observation without a centroid jump.
# Заданное смещение должно сохраняться при исчезновении точки, без скачка центра.
def check_mass_translation():
    global players, players_seen, track_teams, keeper_proxy, BLOCKS, defending_left
    saved=(players,players_seen,track_teams,keeper_proxy,BLOCKS,defending_left)
    try:
        track_teams=np.array([0]*5+[1]*5)
        start=np.array([[20.+i*2,20.+i] for i in range(5)]+[[70.+i*2,20.+i] for i in range(5)])
        players=np.repeat(start[None],8,axis=0)
        players[:,:,0]+=np.arange(8)[:,None]*.01
        players[4:,4]=np.nan
        players_seen=np.isfinite(players).all(-1)
        keeper_proxy=np.zeros((8,10),bool)
        BLOCKS=[(1,0,7)];defending_left={1:[True,False]}
        velocity=team_mass_velocity(1,25.,8)
        # Both teams move +0.25 m/s; relative attacking motion is +0.5 m/s.
        # Обе команды движутся вправо на 0,25 м/с; разность в направлениях атаки +0,5.
        np.testing.assert_allclose(velocity[1:],.5,atol=1e-10)
    finally:
        players,players_seen,track_teams,keeper_proxy,BLOCKS,defending_left=saved
check_mass_translation()


In [ ]:
# Only the public labels are used; the held-out minute cannot rank configurations.
# Только публичные метки; проверочная минута не участвует в выборе параметров.
truth_table=pm.read_label_csv(ROOT/"evaluation/gt/BAR-ATH_possession_gt_public.csv")
truth=pm.label_codes(truth_table)
public=np.flatnonzero(truth!=pm.X)
assert len(public)==7500 and np.all(np.diff(public)==1)
START,END=int(public[0]),int(public[-1]+1)
SPLIT=START+240*FPS
TRAIN=slice(START,SPLIT); VALIDATION=slice(SPLIT,END)

def score(labels, selection):
    true=truth[selection]; pred=pm.label_codes(labels[selection])
    cm=pm.scored_confusion(pm.confusion(true,pred)); runs=pm.valid_runs(true)
    spell=pm.spell_scores(true,pm.fill_abstentions(pred,runs),runs)
    return dict(macroF1=pm.macro_f1(cm),spellF1=spell['mean_f1'],
                accuracy=float(np.mean(true==pred)),predicted_possessions=spell['n_pred'],
                true_possessions=spell['n_gt'],X_fraction=float(np.mean(pred==pm.X)),
                H_F1=pm.per_class_prf(cm,0)[2],A_F1=pm.per_class_prf(cm,1)[2],D_F1=pm.per_class_prf(cm,2)[2])

source_signature=dict(atd_sha256=sha256(PATHS['atd_raw']),meta_sha256=sha256(PATHS['atd_meta']),
                      tagging_sha256=sha256(PATHS['smart_tagging']),
                      public_gt_sha256=sha256(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv'))
save_json(OUT/'run_metadata.json',dict(sources=source_signature,grid=GRID,
          train_frames=[START,SPLIT-1],validation_frames=[SPLIT,END-1],
          baseline_dead_config=CFG,environment={k:importlib.metadata.version(k) for k in ['numpy','pandas','matplotlib','tqdm']}))


## Перебор
Подбираем лаг измерения движения, масштаб пространственных весов, окно сглаживания, порог скорости и длительность подтверждения.

In [ ]:
# Every grid value is declared in grid.json before evaluating labels.
# Все варианты заранее перечислены в grid.json, включая порядок разрешения ничьих.
keys=list(GRID)
configs=[dict(zip(keys,values)) for values in itertools.product(*(GRID[k] for k in keys))]
motion_cache={}; smooth_cache={}; rows=[]
for index,cfg in enumerate(tqdm(configs,desc="Mass grid / Перебор массы")):
    key=(cfg['lag_frames'],cfg['weight_scale_m'])
    if key not in motion_cache: motion_cache[key]=team_mass_velocity(*key,END)
    smooth_key=(*key,cfg['window_frames'])
    if smooth_key not in smooth_cache:
        smooth_cache[smooth_key]=trailing_mean(motion_cache[key],cfg['window_frames'],dead,BLOCKS)
    labels=decode_signal(smooth_cache[smooth_key],dead,BLOCKS,cfg['threshold_mps'],cfg['confirm_frames'])
    rows.append(dict(candidate=index,**cfg,**score(labels,TRAIN)))
    if index%10==0:
        save_json(OUT/'progress.json',dict(stage='grid',done=index+1,total=len(configs)))
        pd.DataFrame(rows).to_csv(OUT/'grid_train.csv',index=False)
table=pd.DataFrame(rows).sort_values(['spellF1','macroF1','candidate'],ascending=[False,False,True])
table.to_csv(OUT/'grid_train.csv',index=False)
selected=configs[int(table.iloc[0].candidate)]
save_json(OUT/'selected_config.json',selected)
print('Selected on training only:',selected)


## Фиксируем выбранный вариант и считаем весь матч

In [ ]:
save_json(OUT/'progress.json',dict(stage='full_match'))
raw=team_mass_velocity(selected['lag_frames'],selected['weight_scale_m'],N)
signal=trailing_mean(raw,selected['window_frames'],dead,BLOCKS)
labels=decode_signal(signal,dead,BLOCKS,selected['threshold_mps'],selected['confirm_frames'])
# The selected full-match prediction must match the prefix used in the search.
# Полный расчёт выбранного варианта должен совпасть с префиксом из перебора.
full_train=score(labels,TRAIN)
assert np.isclose(full_train['spellF1'],table.iloc[0].spellF1)
assert np.isclose(full_train['macroF1'],table.iloc[0].macroF1)
result=pd.DataFrame({'frame':np.arange(N),'label':labels})
error=pm.validate_submission(result,N)
assert error is None,error
result.to_csv(OUT/'submission.csv',index=False)
report=dict(train=full_train,validation=score(labels,VALIDATION),public_all=score(labels,slice(START,END)))
save_json(OUT/'metrics.json',report)
comparison=[]
for name,path in [('001_box5_vote100',ROOT/'local_review/experiments/001_box5_vote100/box5_vote100.csv'),
                  ('003_team_mass',ROOT/'local_review/experiments/003_team_mass/submission.csv'),
                  ('004_mass_near_ball',ROOT/'local_review/experiments/004_mass_near_ball/submission.csv')]:
    if path.exists():
        frame=pm.read_label_csv(path)
        assert pm.validate_submission(frame,N) is None
        predicted=frame.label.to_numpy()
        comparison.append(dict(experiment=name,**score(predicted,VALIDATION)))
pd.DataFrame(comparison).to_csv(OUT/'comparison_validation.csv',index=False)
save_json(OUT/'progress.json',dict(stage='completed',grid_candidates=len(configs),
          seconds=time.perf_counter()-started,submission_sha256=sha256(OUT/'submission.csv')))
# Aggregate diagnostics only / Только агрегаты, не исходные координаты
print(json.dumps(report,indent=2))
print('Output:',OUT/'submission.csv')


In [ ]:
# Rank curve shows sensitivity on training data, not a validation leaderboard.
# Кривая рангов показывает чувствительность на обучении, а не рейтинг по проверке.
fig,ax=plt.subplots(figsize=(9,3))
ax.plot(np.arange(1,len(table)+1),table.spellF1.to_numpy())
ax.set(xlabel='Configuration rank / Ранг варианта',ylabel='Training spellF1',title=EXPERIMENT)
ax.grid(alpha=.25);fig.tight_layout();fig.savefig(OUT/'grid_train_curve.png',dpi=140)
plt.show()
print(pd.DataFrame(comparison).to_string(index=False))
